# Etapa 1: Instalar as dependências

In [17]:
%pip install pandas pyarrow azure-storage-file-datalake

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Etapa 2: Importações 

In [18]:
import sqlite3
import pandas as pd
import os
from azure.storage.filedatalake import DataLakeServiceClient

# Etapa 3: Função para exportar tabelas do SQLite para CSV

In [31]:
def export_sqlite_to_csv(db_path: str, export_dir: str, mode: str = 'skip'):
    """
    Exporta todas as tabelas do banco SQLite para arquivos CSV.
    mode='skip': pula exportação se o arquivo CSV já existe.
    mode='overwrite': sobrescreve arquivos CSV existentes.
    mode='force': deleta arquivos CSV antes de exportar.
    """
    print(f"[Exportação] Modo selecionado: {mode}")
    os.makedirs(export_dir, exist_ok=True)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    for table in tables:
        csv_path = os.path.join(export_dir, f"{table}.csv")
        if mode == 'skip' and os.path.exists(csv_path):
            print(f"Arquivo já existe, pulando exportação: {csv_path}")
            continue
        elif mode == 'force' and os.path.exists(csv_path):
            os.remove(csv_path)
            print(f"Arquivo antigo deletado: {csv_path}")
        # Para 'overwrite' e 'force', sempre exporta
        df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
        df.to_csv(csv_path, index=False)
        print(f"Exportado: {csv_path}")
    print("✅ Exportação concluída")
    conn.close()

# Etapa 4: Função para criar o cliente do Data Lake Gen2 usando SAS Token

In [33]:
def create_datalake_client(account_name: str, sas_token: str, filesystem_name: str):
    """
    Cria e retorna o filesystem_client do Data Lake Gen2.
    """
    service_client = DataLakeServiceClient(
        account_url=f"https://{account_name}.dfs.core.windows.net",
        credential=sas_token
    )
    filesystem_client = service_client.get_file_system_client(filesystem_name)
    return filesystem_client


# Etapa 5: Função para upload dos arquivos CSV para o Data Lake Gen2 (pula se já existe no servidor)

In [32]:
def upload_files_to_datalake(local_dir: str, landing_zone_path: str, filesystem_client, mode: str = 'skip'):
    """
    Faz upload dos arquivos de local_dir para o Data Lake Gen2.
    mode='skip': pula arquivos que já existem no Data Lake.
    mode='overwrite': sobrescreve arquivos existentes.
    mode='force': deleta arquivos remotos antes de subir.
    """
    print(f"[Upload] Modo selecionado: {mode}")
    for filename in os.listdir(local_dir):
        file_path = os.path.join(local_dir, filename)
        remote_path = f"{landing_zone_path}/{filename}" if landing_zone_path else filename
        file_client = filesystem_client.get_file_client(remote_path)
        if mode == 'skip':
            try:
                file_client.get_file_properties()
                print(f"Arquivo já existe no Data Lake, pulando upload: {remote_path}")
                continue
            except Exception:
                pass
        elif mode == 'force':
            try:
                file_client.delete_file()
                print(f"Arquivo antigo deletado no Data Lake: {remote_path}")
            except Exception:
                pass
        # Para 'overwrite' e 'force', sempre faz upload
        with open(file_path, "rb") as data:
            file_client.upload_data(data, overwrite=True)
        print(f"Upload realizado: {remote_path}")
    print("✅ Upload concluído")









# Uso

In [35]:
db_path = '../data/db.sqlite'  # Caminho para o banco SQLite
export_dir = '../data/exported_tables'  # Diretório para salvar os CSVs

# Parâmetros do Data Lake
account_name = "datalake3a8889b275cb1ce3"
sas_token = "sv=2024-11-04&ss=bfqt&srt=co&sp=rwdlacupyx&se=2025-06-27T17:02:45Z&st=2025-06-27T09:02:45Z&spr=https&sig=MyL5Q1vHM3QOKALoex%2BiitnSff8TRVGMTPpL6z2Vzzs%3D"
filesystem_name = "landing-zone"  # Nome do container
landing_zone_path = "csvs"  # Pasta de destino no Data Lake (ou "" para raiz)

export_mode = 'force'      # Para exportação do SQLite
upload_mode = 'skip'      # Para upload ao Data Lake

# 1. Exporta tabelas do SQLite para CSV
export_sqlite_to_csv(db_path, export_dir, mode=export_mode)

# 2. Cria o cliente do Data Lake
datalake_client = create_datalake_client(account_name, sas_token, filesystem_name)

# 3. Faz upload dos arquivos CSV para o Data Lake
upload_files_to_datalake(export_dir, landing_zone_path, datalake_client, mode=upload_mode)

[Exportação] Modo selecionado: force
Arquivo antigo deletado: ../data/exported_tables\product_category_name_translation.csv
Exportado: ../data/exported_tables\product_category_name_translation.csv
Arquivo antigo deletado: ../data/exported_tables\sellers.csv
Exportado: ../data/exported_tables\sellers.csv
Arquivo antigo deletado: ../data/exported_tables\customers.csv
Exportado: ../data/exported_tables\customers.csv
Arquivo antigo deletado: ../data/exported_tables\geolocation.csv
Exportado: ../data/exported_tables\geolocation.csv
Arquivo antigo deletado: ../data/exported_tables\order_items.csv
Exportado: ../data/exported_tables\order_items.csv
Arquivo antigo deletado: ../data/exported_tables\order_payments.csv
Exportado: ../data/exported_tables\order_payments.csv
Arquivo antigo deletado: ../data/exported_tables\order_reviews.csv
Exportado: ../data/exported_tables\order_reviews.csv
Arquivo antigo deletado: ../data/exported_tables\orders.csv
Exportado: ../data/exported_tables\orders.csv
Arq